[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-rule-based.ipynb)

# Rule-Based Classifiers — RIPPER & CN2

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

A niche but genuinely still-used family: classifiers that output a small, human-readable list of IF-THEN rules instead of a tree, a hyperplane, or an ensemble of black boxes.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Packages that Colab does not ship by default (a no-op if already installed)
%pip install -q wittgenstein

The lesson's cells assume a loan-approval table `df` (income_lakhs, cibil_score, existing_loans, employment, approved). We build a reproducible one here.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(11)
n = 600
df = pd.DataFrame({
    'income_lakhs': rng.uniform(15, 80, n).round(1),
    'cibil_score': rng.integers(540, 800, n),
    'existing_loans': rng.integers(0, 5, n),
    'employment': rng.choice(['salaried', 'self_employed', 'contract'], n, p=[0.6, 0.25, 0.15]),
})
approve = ((df.income_lakhs > 40) & (df.cibil_score > 660) & (df.existing_loans < 3)) | (df.income_lakhs > 65)
flip = rng.random(n) < 0.05                       # 5% label noise, as in real data
df['approved'] = (approve ^ flip).astype(int)
print(df.shape, "approval rate:", round(df.approved.mean(), 3))
df.head()

> **⚠ Why This Page Is Marked "Advanced" (and Niche)**
>
> Rule-induction algorithms are not a mainstream default the way Random Forest or XGBoost are — most modern tabular problems are better served by ensemble methods covered earlier in this course. This page is included because **rule sets remain in genuine use** specifically where interpretability is a hard requirement (medical decision support, enterprise business-rule automation, regulatory/compliance settings) rather than a nice-to-have — a narrower niche than the rest of this course, introduced here at a foundational level.

## Sequential Covering — Learning One Rule at a Time

Decision Trees (covered earlier) learn all their splits simultaneously via one recursive partitioning process, and rules are only implicit — each root-to-leaf path *is* a rule, but the tree structure itself is the primary object. Rule-induction algorithms flip this: rules are the primary object, learned one at a time via **sequential covering**:

$$\text{Repeat: learn the single best rule covering as many remaining positive examples (and as few negatives) as possible} \to \text{remove the examples it covers} \to \text{repeat on what's left, until no positive examples remain uncovered}$$

Each rule is grown by greedily adding conditions (feature thresholds or category matches) that most increase the rule's precision on the training data, then pruned back to avoid overfitting to noise. The result is a compact, ordered or unordered list of IF-THEN rules — typically far shorter and more directly readable than an unrolled decision tree, especially once the tree gets deep.

## RIPPER in Practice — A Loan-Approval Ruleset

**RIPPER** (Repeated Incremental Pruning to Produce Error Reduction) is the best-known modern sequential-covering algorithm, and remains a standard reference point in interpretable-ML literature. Applied to a synthetic HDFC Bank-style loan approval dataset (800 applicants; approval genuinely driven by CIBIL score, income, and existing loan count, with 5% label noise):

In [ ]:
import wittgenstein as lw
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# df columns: income_lakhs, cibil_score, existing_loans, employment, approved
train, test = train_test_split(df, test_size=0.25, random_state=11, stratify=df['approved'])

ripper = lw.RIPPER(random_state=11, n_discretize_bins=4, max_rules=5)
ripper.fit(train, class_feat='approved', pos_class=1)
print(ripper.ruleset_)

preds = ripper.predict(test.drop('approved', axis=1))
print(f"Accuracy:  {accuracy_score(test['approved'], preds):.4f}")
print(f"Precision: {precision_score(test['approved'], preds):.4f}")
print(f"Recall:    {recall_score(test['approved'], preds):.4f}")
print(f"F1:        {f1_score(test['approved'], preds):.4f}")

Read as plain English, the last (broadest) rule alone says: *"IF cibil_score is between 695.81 and 776.67 AND existing_loans is fewer than 1, THEN approve."* An underwriter can read, question, and directly override any single rule in this list — something not meaningfully possible with a Random Forest's 100+ averaged trees or XGBoost's boosted ensemble, even though those models would likely score higher accuracy on this same data.

## Where This Still Matters in Industry

Sequential-covering rule induction (RIPPER, CN2, and related methods) continues to appear in three recurring contexts: interpretable-ML reference material and benchmarking (it's a standing chapter in modern interpretable-ML literature, used as a baseline against black-box explainability techniques like SHAP covered on the Model Interpretability page); enterprise rule-generation automation, where organizations that previously relied on manually written business rules use rule-induction to propose candidate rules for human review rather than replacing manual rules outright; and explainability-critical applied research, including medical decision-support studies that specifically favor RIPPER's directly-readable output over a higher-scoring but opaque alternative when clinician trust is part of the deployment requirement.

## Trade-offs vs. Decision Trees

| Aspect | Decision Tree | Rule-Based (RIPPER/CN2) |
|---|---|---|
| Learned jointly or one at a time? | All splits learned jointly in one recursive process | One rule at a time, via sequential covering |
| Readability at scale | Degrades as the tree gets deep — a path with 10 splits is 10 conditions to read | Rules stay compact by construction; each rule typically has few conditions |
| Handles a "default" case | Every leaf is reachable and labelled by construction | Needs an explicit default class for examples matching no rule |
| Raw accuracy on large tabular data | Usually decent alone; excellent as a Random Forest/XGBoost ensemble | Typically similar to or slightly below a single Decision Tree |
| Best used when | General-purpose classification/regression | Interpretability is a hard requirement, not just a preference |

## Try It — Test Your Own Applicant Against the Ruleset

Adjust the four inputs below and see which of the exact 5 rules from the RIPPER output fires — or whether the applicant falls through to the default reject. This is exactly what "an underwriter can read, question, and directly override any single rule" looks like in practice.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Coverage and precision of one rule

A rule fires when `income > 40 and cibil > 650`. On the small table below, store its **coverage** (share of rows it fires on) in `coverage` and its **precision** (share of fired rows that are truly approved) in `precision`.

In [ ]:
import pandas as pd
t = pd.DataFrame({"income": [50, 30, 45, 60, 20, 42], "cibil": [700, 720, 640, 680, 600, 660], "approved": [1, 0, 0, 1, 0, 0]})
coverage = precision = None   # TODO


In [ ]:
try:
    check("coverage 3/6", coverage == 0.5)
    check("precision 2/3", round(precision, 4) == 0.6667)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
t = pd.DataFrame({"income": [50, 30, 45, 60, 20, 42], "cibil": [700, 720, 640, 680, 600, 660], "approved": [1, 0, 0, 1, 0, 0]})
fired = (t.income > 40) & (t.cibil > 650)
coverage = fired.mean()
precision = t.loc[fired, "approved"].mean()

```

</details>

### Exercise 2 · Medium · A decision list (first match wins)

Write `predict_row(row, rules, default)` where `rules` is a list of `(condition_function, label)` pairs checked in order; return the label of the first rule whose condition is true, otherwise `default`.

In [ ]:
def predict_row(row, rules, default):
    pass   # TODO


In [ ]:
try:
    rules = [(lambda r: r["income"] > 60, 1), (lambda r: r["cibil"] < 600, 0)]
    check("first rule wins", predict_row({"income": 70, "cibil": 500}, rules, 0) == 1)
    check("second rule", predict_row({"income": 30, "cibil": 500}, rules, 1) == 0)
    check("default", predict_row({"income": 30, "cibil": 700}, rules, 1) == 1)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
def predict_row(row, rules, default):
    for cond, label in rules:
        if cond(row):
            return label
    return default

```

</details>

### Exercise 3 · Stretch · Turn a tree into readable rules

Fit a `DecisionTreeClassifier(max_depth=3, random_state=0)` on `income_lakhs`, `cibil_score`, `existing_loans` of the lesson's `df`. Store the IF-THEN text from `sklearn.tree.export_text` in `rules_text` and the number of leaves (one rule each) in `n_rules`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text
features = ["income_lakhs", "cibil_score", "existing_loans"]
rules_text = n_rules = None   # TODO


In [ ]:
try:
    check("text mentions a feature", "income_lakhs" in rules_text or "cibil_score" in rules_text)
    check("at most 8 rules", 2 <= n_rules <= 8)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.tree import DecisionTreeClassifier, export_text
features = ["income_lakhs", "cibil_score", "existing_loans"]
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(df[features], df["approved"])
rules_text = export_text(tree, feature_names=features)
n_rules = int(tree.get_n_leaves())

```

Every root-to-leaf path is an IF-THEN rule; regulators and loan officers can read and audit them.

</details>

---
*Back to the course: **Machine Learning End To End → Rule-Based Classifiers — RIPPER & CN2**.*